In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pykep

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

from orbital_mechanics.solar_system import SolarSystem
from common.constants import ALTAIRA_MU as MU, ALTAIRA_AU as AU, YEAR
from common.constants import solar_C, solar_r0, solar_A, sc_mass
from common.solar_sail import calc_opt_un

In [91]:
# initial state of spacecraft

r0,v0 = pykep.par2ic(np.array([8.76956*AU, 0.99886, 0.0, 0.0, 0, 0.0]), MU)

r0 = np.array(r0); v0 = np.array(v0)
print(r0,v0)

energy0 = np.linalg.norm(v0)**2/2 - MU/np.linalg.norm(r0)
print(energy0)

[1495574.55330258       0.               0.        ] [  0.         431.55645966   0.        ]
-53.108950794965494


In [92]:
def dydt(_, y):
    r = y[0:3]; v = y[3:6]

    # calculate control vector
    ur = -r; ud = v
    ur = ur/np.linalg.norm(ur);
    ud = ud/np.linalg.norm(ud)
    un = calc_opt_un(ur,ud)

    # calculate the force of the solar sail
    r_mag = np.linalg.norm(r)
    s_mag = 2*(solar_C*solar_A/sc_mass) * (solar_r0/r_mag)**2 * np.dot(ur,un)**2
    s_mag = s_mag / 1000    # convert from m/s^2 to km/s^2
    s_acc = -s_mag * un   # the acceleration acts opposite to un

    dy = np.empty((6,), dtype=np.float64)
    dr = v; dv = (-MU/r_mag**3 * r) + s_acc   # gravity and solar acceleration
    dy[0:3] = dr; dy[3:6] = dv

    return dy

In [93]:
dydt(_, np.concatenate([r0,v0]))

array([ 0.00000000e+00,  4.31556460e+02,  0.00000000e+00, -6.11528501e-02,
        1.14668402e-03, -0.00000000e+00])

In [94]:
# event function to propagate it only till 1 AU distance
def event(_, y):
    r = y[0:3]
    r_mag = np.linalg.norm(r)
    return r_mag - 1*AU

event.terminal = True

In [95]:
res = solve_ivp(dydt, [0, 200*YEAR], np.concatenate([r0,v0]), events=event)

In [99]:
# calculate the final energy
final_rv = res['y'][:,-1].squeeze()
rf = final_rv[0:3]; vf = final_rv[3:6]
rf_mag = np.linalg.norm(rf); vf_mag = np.linalg.norm(vf)

Ef = vf_mag**2/2 - MU/rf_mag
print(f"change in energy: {Ef} km^2/s^2, required around 200 km^2/s^2")

change in energy: 6262.364642316923 km^2/s^2, required around 200 km^2/s^2


In conclusion, the solar sail is:
1. ridiculously overpowered in the inner system (<Eden) (thank the Oberth effect)
2. medium power in the asteroid belt
3. not powerful in the outer solar system